# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [2]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [3]:
from dotenv import load_dotenv

load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://185.50.38.163:33014")
MLFLOW_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_amirhossein_sa
Experiment ID: 31


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [4]:
DATASET_VERSION = "v1_student"
FEATURE_DIR = Path("../week2-01/data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
    loaded_dataset_path = parquet_path
else:
    feature_df = pd.read_csv(csv_path)
    loaded_dataset_path = csv_path

metadata = {}
if metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)

print("Loaded:", loaded_dataset_path)
print("Shape:", feature_df.shape)
feature_df.head()

Loaded: ../week2-01/data/features/listing_availability_features_v1_student.csv
Shape: (10480, 34)


,listing_id,cutoff_date,dataset_version,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,...,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,has_reviews_before_cutoff
0,27886,2025-12-10,v1_student,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,...,30.0,10,0.333333,3.000000,30.0,30,1,0.033333,1,1
1,28871,2025-12-10,v1_student,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,...,730.0,8,0.266667,1.866667,730.0,30,4,0.133333,1,1
2,29051,2025-12-10,v1_student,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,...,730.0,12,0.400000,2.000000,730.0,30,0,0.000000,1,1
3,44391,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,...,730.0,0,0.000000,3.000000,730.0,30,0,0.000000,1,1
4,48373,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,...,1125.0,0,0.000000,3.000000,1125.0,30,0,0.000000,1,1


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [5]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

if TARGET_COL not in feature_df.columns:
    raise KeyError(f"Target column {TARGET_COL!r} is missing from the dataset.")
if feature_df[TARGET_COL].isna().any():
    raise ValueError(f"Target column {TARGET_COL!r} contains missing values.")

y = feature_df[TARGET_COL].astype("int64")
if not set(y.unique()).issubset({0, 1}):
    raise ValueError(f"{TARGET_COL!r} must be binary with values 0 and 1.")

forbidden_set = set(FORBIDDEN_MODEL_COLUMNS)
clean_feature_cols = [col for col in feature_df.columns if col not in forbidden_set]
X_clean = feature_df.loc[:, clean_feature_cols].copy()

# raise NotImplementedError("Create y, clean_feature_cols, and X_clean.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.331584
1    0.668416
Name: proportion, dtype: float64
Clean feature count: 27
['room_type', 'property_type', 'neighbourhood_name', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d', 'has_reviews_before_cutoff']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [6]:
LEAKAGE_COLUMN = "future_available_rate_30d"

if LEAKAGE_COLUMN not in feature_df.columns:
    raise KeyError(f"Leakage demonstration column {LEAKAGE_COLUMN!r} is missing.")

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]
X_leaky = feature_df.loc[:, leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 28
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 27)
Test shape: (2096, 27)
Train target rate: 0.6684160305343512
Test target rate: 0.6684160305343512


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [8]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def get_column_groups(X):
    """Return numeric and categorical column names for a feature frame."""
    numeric = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [col for col in X.columns if col not in numeric]
    return numeric, categorical


def build_preprocessor(X):
    """Build a fresh preprocessor for exactly the columns in X."""
    numeric, categorical = get_column_groups(X)
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric),
            ("categorical", categorical_transformer, categorical),
        ],
        remainder="drop",
    )


numeric_cols, categorical_cols = get_column_groups(X_clean)
preprocessor = build_preprocessor(X_clean)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 22
Categorical columns: 5


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [9]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be between 0 and 1.")

    y_score = np.asarray(get_positive_scores(model, X_test), dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_score)),
    }
    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [10]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
MODEL_DIR = Path("outputs/mlflow_models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    run_artifact_dir = ARTIFACT_DIR / run_name
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_true, y_pred),
        display_labels=[0, 1],
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix: {run_name}")
    fig.tight_layout()
    fig.savefig(run_artifact_dir / "confusion_matrix.png", dpi=150)
    plt.close(fig)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    with (run_artifact_dir / "classification_report.json").open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    with (run_artifact_dir / "feature_columns.json").open("w", encoding="utf-8") as f:
        json.dump(list(feature_cols), f, indent=2)
    with (run_artifact_dir / "dataset_metadata_snapshot.json").open("w", encoding="utf-8") as f:
        json.dump(metadata or {}, f, indent=2, default=str)

    return run_artifact_dir

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [11]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    pipeline.fit(X_train, y_train)
    metrics, y_pred, _ = evaluate_binary_classifier(
        pipeline, X_test, y_test, threshold=threshold
    )
    run_artifact_dir = save_run_artifacts(
        run_name, y_test, y_pred, feature_cols, metadata
    )

    logged_params = {
        "dataset_version": DATASET_VERSION,
        "feature_count": len(feature_cols),
        "test_size": 0.20,
        "threshold": threshold,
        **model_params,
    }
    logged_params = {
        key: "None" if value is None else value for key, value in logged_params.items()
    }
    run_tags = {
        "dataset_version": DATASET_VERSION,
        "notebook": "02_mlflow_experiments_student",
        "pipeline_logged": "true",
        **tags,
    }

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(logged_params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(run_tags)
        mlflow.log_artifacts(str(run_artifact_dir), artifact_path="evaluation")

        # MLflow 3 clients use a logged-model endpoint that older servers may not have.
        # Save the standard MLflow model locally, then upload it with the compatible
        # run-artifact API. This still stores the complete sklearn Pipeline.
        model_dir = MODEL_DIR / run.info.run_id
        if model_dir.exists():
            shutil.rmtree(model_dir)
        mlflow.sklearn.save_model(pipeline, path=str(model_dir))
        mlflow.log_artifacts(str(model_dir), artifact_path="model")

        result = {
            "run_id": run.info.run_id,
            "run_name": run_name,
            **metrics,
        }

    print(run_name, result)
    return result

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [12]:
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_leaky)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

v0_result = run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=leaky_pipeline,
    X_train=X_leaky_train,
    X_test=X_leaky_test,
    y_train=y_leaky_train,
    y_test=y_leaky_test,
    feature_cols=leaky_feature_cols,
    model_params={"max_iter": 1000, "random_state": RANDOM_STATE},
    tags={
        "leakage_status": "leaky",
        "known_defect": "uses future_available_rate_30d",
        "model_family": "logistic_regression",
        "run_type": "leakage_demo",
    },
)

2026/06/07 11:03:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/31/runs/fb59dacccc514f348dcdc44b653fca52
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v0_leaky_logistic_regression {'run_id': 'fb59dacccc514f348dcdc44b653fca52', 'run_name': 'v0_leaky_logistic_regression', 'accuracy': 0.9928435114503816, 'precision': 0.9907932011331445, 'recall': 0.9985724482512491, 'f1': 0.9946676146462851, 'roc_auc': 0.9998377315278398}


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [13]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_clean)),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

v1_result = run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"strategy": "most_frequent"},
    tags={
        "leakage_status": "clean",
        "model_family": "dummy",
        "run_type": "baseline",
    },
)

2026/06/07 11:04:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/31/runs/b933a411a3e049cfaeff0032a92d53f2
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v1_dummy_baseline {'run_id': 'b933a411a3e049cfaeff0032a92d53f2', 'run_name': 'v1_dummy_baseline', 'accuracy': 0.6684160305343512, 'precision': 0.6684160305343512, 'recall': 1.0, 'f1': 0.8012582213325707, 'roc_auc': 0.5}


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [14]:
clean_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_clean)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

v2_result = run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=clean_logistic_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"max_iter": 1000, "class_weight": None, "random_state": RANDOM_STATE},
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
        "run_type": "clean_model",
    },
)

2026/06/07 11:05:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/31/runs/f6bbc00392bd4c3ea3af9d7c7c99bb33
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v2_clean_logistic_regression {'run_id': 'f6bbc00392bd4c3ea3af9d7c7c99bb33', 'run_name': 'v2_clean_logistic_regression', 'accuracy': 0.8797709923664122, 'precision': 0.8997912317327766, 'recall': 0.9229122055674518, 'f1': 0.9112050739957717, 'roc_auc': 0.9315370829674592}


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [15]:
balanced_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_clean)),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

v3_result = run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=balanced_logistic_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
    },
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
        "run_type": "class_weight_tuning",
    },
)

2026/06/07 11:06:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/31/runs/45b69b782c8f4496bde936d2504fff17
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v3_balanced_logistic_regression {'run_id': '45b69b782c8f4496bde936d2504fff17', 'run_name': 'v3_balanced_logistic_regression', 'accuracy': 0.8807251908396947, 'precision': 0.9173313995649021, 'recall': 0.9029264810849393, 'f1': 0.9100719424460432, 'roc_auc': 0.9321789677465735}


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [16]:
threshold_results = []
for threshold in [0.30, 0.40, 0.50, 0.60]:
    threshold_pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(X_clean)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )
    result = run_mlflow_experiment(
        run_name=f"v4_logistic_threshold_{threshold:.2f}",
        pipeline=threshold_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "max_iter": 1000,
            "class_weight": None,
            "random_state": RANDOM_STATE,
        },
        tags={
            "leakage_status": "clean",
            "model_family": "logistic_regression",
            "run_type": "threshold_tuning",
        },
        threshold=threshold,
    )
    threshold_results.append(result)

pd.DataFrame(threshold_results).sort_values("f1", ascending=False)

2026/06/07 11:06:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v4_logistic_threshold_0.30 at: http://185.50.38.163:33014/#/experiments/31/runs/9a2a7110a48145bf8e062b882fdfeb70
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v4_logistic_threshold_0.30 {'run_id': '9a2a7110a48145bf8e062b882fdfeb70', 'run_name': 'v4_logistic_threshold_0.30', 'accuracy': 0.8764312977099237, 'precision': 0.8811748998664887, 'recall': 0.9421841541755889, 'f1': 0.9106588478785789, 'roc_auc': 0.9315370829674592}


2026/06/07 11:06:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v4_logistic_threshold_0.40 at: http://185.50.38.163:33014/#/experiments/31/runs/5a22fde9123e443997ef2db64526ac8e
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v4_logistic_threshold_0.40 {'run_id': '5a22fde9123e443997ef2db64526ac8e', 'run_name': 'v4_logistic_threshold_0.40', 'accuracy': 0.8797709923664122, 'precision': 0.8926862611073137, 'recall': 0.9321912919343326, 'f1': 0.9120111731843575, 'roc_auc': 0.9315370829674592}


2026/06/07 11:06:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v4_logistic_threshold_0.50 at: http://185.50.38.163:33014/#/experiments/31/runs/3a14cbd427bc48bea62f4ace4ba80e6e
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v4_logistic_threshold_0.50 {'run_id': '3a14cbd427bc48bea62f4ace4ba80e6e', 'run_name': 'v4_logistic_threshold_0.50', 'accuracy': 0.8797709923664122, 'precision': 0.8997912317327766, 'recall': 0.9229122055674518, 'f1': 0.9112050739957717, 'roc_auc': 0.9315370829674592}


2026/06/07 11:06:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v4_logistic_threshold_0.60 at: http://185.50.38.163:33014/#/experiments/31/runs/4ea7e37e57ba41d090d548d91b503a9c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v4_logistic_threshold_0.60 {'run_id': '4ea7e37e57ba41d090d548d91b503a9c', 'run_name': 'v4_logistic_threshold_0.60', 'accuracy': 0.8821564885496184, 'precision': 0.9103840682788051, 'recall': 0.913633119200571, 'f1': 0.9120057000356252, 'roc_auc': 0.9315370829674592}


,run_id,run_name,accuracy,precision,recall,f1,roc_auc
1,5a22fde9123e443997ef2db64526ac8e,v4_logistic_threshold_0.40,0.879771,0.892686,0.932191,0.912011,0.931537
3,4ea7e37e57ba41d090d548d91b503a9c,v4_logistic_threshold_0.60,0.882156,0.910384,0.913633,0.912006,0.931537
2,3a14cbd427bc48bea62f4ace4ba80e6e,v4_logistic_threshold_0.50,0.879771,0.899791,0.922912,0.911205,0.931537
0,9a2a7110a48145bf8e062b882fdfeb70,v4_logistic_threshold_0.30,0.876431,0.881175,0.942184,0.910659,0.931537


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [17]:
random_forest_params = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
}
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_clean)),
        (
            "classifier",
            RandomForestClassifier(**random_forest_params, n_jobs=-1),
        ),
    ]
)

v5_result = run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=random_forest_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={**random_forest_params, "n_jobs": -1},
    tags={
        "leakage_status": "clean",
        "model_family": "random_forest",
        "run_type": "clean_model",
    },
)

2026/06/07 11:06:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/31/runs/d97de4e1b1334f1a8996b84d582616f6
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/31
v5_random_forest {'run_id': 'd97de4e1b1334f1a8996b84d582616f6', 'run_name': 'v5_random_forest', 'accuracy': 0.8936068702290076, 'precision': 0.9362962962962963, 'recall': 0.9022127052105638, 'f1': 0.9189385677935297, 'roc_auc': 0.961253780701349}


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [18]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experiment {EXPERIMENT_NAME!r} does not exist.")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC", "metrics.roc_auc DESC"],
)
comparison_columns = {
    "run_id": "run_id",
    "tags.mlflow.runName": "run_name",
    "tags.leakage_status": "leakage_status",
    "tags.model_family": "model_family",
    "metrics.accuracy": "accuracy",
    "metrics.precision": "precision",
    "metrics.recall": "recall",
    "metrics.f1": "f1",
    "metrics.roc_auc": "roc_auc",
}
comparison_df = (
    runs_df.reindex(columns=comparison_columns)
    .rename(columns=comparison_columns)
    .sort_values(["f1", "roc_auc"], ascending=False, na_position="last")
    .reset_index(drop=True)
)

comparison_df

,run_id,run_name,leakage_status,model_family,accuracy,precision,recall,f1,roc_auc
0,fb59dacccc514f348dcdc44b653fca52,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
1,c4168a7f579d4b4494cb78b6102a100d,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
2,02bbcc67c5a4421ea0035fc1dddbf029,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
3,cfc879c3b4f847dab1a86f9f97792d12,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
4,2915110084054a91b380c4a088efd73b,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
5,44e0fb8942974ab087e57528279ed699,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
6,fff5a1e486a948c7bb883ea4efae6647,v0_leaky_logistic_regression,leaky,logistic_regression,0.992844,0.990793,0.998572,0.994668,0.999838
7,d97de4e1b1334f1a8996b84d582616f6,v5_random_forest,clean,random_forest,0.893607,0.936296,0.902213,0.918939,0.961254
8,5a22fde9123e443997ef2db64526ac8e,v4_logistic_threshold_0.40,clean,logistic_regression,0.879771,0.892686,0.932191,0.912011,0.931537
9,4ea7e37e57ba41d090d548d91b503a9c,v4_logistic_threshold_0.60,clean,logistic_regression,0.882156,0.910384,0.913633,0.912006,0.931537


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [19]:
clean_candidates = comparison_df.loc[
    (comparison_df["leakage_status"] == "clean")
    & (comparison_df["model_family"] != "dummy")
].dropna(subset=["f1", "roc_auc"])
if clean_candidates.empty:
    raise ValueError("No clean real-model runs are available for selection.")

best_row = clean_candidates.sort_values(
    ["f1", "roc_auc", "precision", "recall"],
    ascending=False,
).iloc[0]
BEST_RUN_ID = best_row["run_id"]

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)
print(best_row)

Selected best run: d97de4e1b1334f1a8996b84d582616f6
run_id            d97de4e1b1334f1a8996b84d582616f6
run_name                          v5_random_forest
leakage_status                               clean
model_family                         random_forest
accuracy                                  0.893607
precision                                 0.936296
recall                                    0.902213
f1                                        0.918939
roc_auc                                   0.961254
Name: 7, dtype: object


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [20]:
final_explanation = f"""
I selected {best_row['run_name']} as the final candidate, with F1={best_row['f1']:.3f} and ROC AUC={best_row['roc_auc']:.3f}.
It was the strongest eligible clean real-model run by F1, with ROC AUC and the precision/recall balance used as additional checks.
I rejected the leaky run because it uses future_available_rate_30d, which comes from the label window and would not be available at prediction time.
Next, I would use cross-validation and tune regularization or tree-based hyperparameters before evaluating the final pipeline on a later time window.
""".strip()

print(final_explanation)

I selected v5_random_forest as the final candidate, with F1=0.919 and ROC AUC=0.961.
It was the strongest eligible clean real-model run by F1, with ROC AUC and the precision/recall balance used as additional checks.
I rejected the leaky run because it uses future_available_rate_30d, which comes from the label window and would not be available at prediction time.
Next, I would use cross-validation and tune regularization or tree-based hyperparameters before evaluating the final pipeline on a later time window.
